# Week 83, Day 3: pandas II

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w083d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('orders.csv', 'w', encoding='utf-8') as f:
    f.write('order_id,date,customer,city,product,category,quantity,unit_price\n1001,2026-01-04,Ada Lovelace,York,Notebook,stationery,3,2.50\n1002,2026-01-04,Bo Chen,Leeds,Desk lamp,home,1,24.99\n1003,2026-01-11,Cy Okafor,York,Pens (10),stationery,2,4.20\n1004,2026-01-19,Dee Patel,Hull,Kettle,home,1,32.00\n1005,2026-02-02,Ada Lovelace,York,Mug,home,4,6.75\n1006,2026-02-03,Eli Novak,Bath,Notebook,stationery,10,2.50\n1007,2026-02-14,Bo Chen,Leeds,Headphones,electronics,1,59.00\n1008,2026-02-20,Fay Morgan,Bath,Charger,electronics,2,15.50\n1009,2026-03-01,Cy Okafor,York,Desk lamp,home,1,24.99\n1010,2026-03-07,Gus Adeyemi,Hull,Keyboard,electronics,1,45.00\n1011,2026-03-15,Dee Patel,Hull,Pens (10),stationery,5,4.20\n1012,2026-03-22,Ada Lovelace,York,Headphones,electronics,1,59.00\n1013,2026-03-28,Hal Brooks,Leeds,Mug,home,2,6.75\n1014,2026-04-02,Eli Novak,Bath,Charger,electronics,1,15.50\n')
with open('customers.csv', 'w', encoding='utf-8') as f:
    f.write('customer,joined,tier,email\nAda Lovelace,2025-11-02,gold,ada@example.com\nBo Chen,2025-12-15,silver,bo@example.com\nCy Okafor,2026-01-10,silver,cy@example.com\nDee Patel,2025-06-30,gold,dee@example.com\nEli Novak,2026-02-01,bronze,eli@example.com\nFay Morgan,2026-02-18,bronze,fay@example.com\nGus Adeyemi,2025-09-09,silver,\nHal Brooks,2026-03-20,bronze,hal@example.com\nIvy Shaw,2026-01-25,gold,ivy@example.com\n')

**Predict before you run:** what will this print?



In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
df["total"] = (df["quantity"] * df["unit_price"]).round(2)
by_city = df.groupby("city")["total"].sum().round(2)
print(by_city.to_dict())
print(by_city.idxmax(), df.groupby("city")["order_id"].count().to_dict()["York"])

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
df["total"] = (df["quantity"] * df["unit_price"]).round(2)
summary = (df.groupby("category")
             .agg(orders=("order_id", "count"), revenue=("total", "sum"), biggest=("total", "max"))
             .round(2)
             .reset_index()
             .sort_values("revenue", ascending=False))
print(summary)
print(df.groupby(["city", "category"])["total"].sum().round(2).head(4).to_dict())
df["city_share"] = (df["total"] / df.groupby("city")["total"].transform("sum")).round(2)
print(df[["order_id", "city", "city_share"]].head(3).values.tolist())
print(df.loc[df.groupby("city")["total"].idxmax(), ["city", "order_id"]].values.tolist())

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
orders["total"] = (orders["quantity"] * orders["unit_price"]).round(2)
customers = pd.read_csv("customers.csv")

joined = orders.merge(customers, on="customer", how="left", validate="many_to_one")
print(joined.shape, joined[["order_id", "customer", "tier"]].head(2).values.tolist())
print(joined.groupby("tier")["total"].sum().round(2).to_dict())

who = customers.merge(orders[["customer"]].drop_duplicates(), on="customer", how="left", indicator=True)
print(who.loc[who["_merge"] == "left_only", "customer"].tolist())
both = pd.concat([orders.head(2), orders.tail(1)], ignore_index=True)
print(both["order_id"].tolist())

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
df["total"] = (df["quantity"] * df["unit_price"]).round(2)
df["month"] = df["date"].str[:7]
wide = df.pivot_table(index="month", columns="category", values="total", aggfunc="sum", fill_value=0).round(2)
print(wide)
long = wide.reset_index().melt(id_vars="month", var_name="category", value_name="total")
print(long.shape, long.head(2).values.tolist())

In [ ]:
import pandas as pd

df = pd.read_csv("orders.csv")
df["total"] = (df["quantity"] * df["unit_price"]).round(2)
df["date"] = pd.to_datetime(df["date"])
print(df["date"].dt.month.tolist()[:5], df["date"].dt.day_name().iloc[0])
monthly = df.set_index("date").resample("MS")["total"].sum().round(2)
print([d.strftime("%b") for d in monthly.index], monthly.tolist())
print(monthly.pct_change().round(3).tolist())
gap = df["date"].max() - df["date"].min()
print(gap.days)
print(pd.to_datetime(pd.Series(["15/01/2026", "31/02/2026"]), format="%d/%m/%Y", errors="coerce").isna().tolist())

In [ ]:
import pandas as pd

customers = pd.read_csv("customers.csv")
print(customers.isna().sum().to_dict())
print(len(customers.dropna(subset=["email"])), customers["email"].fillna("(none)").tolist()[6])

scores = pd.Series([80, None, 60, None])
print(scores.mean(), scores.fillna(0).mean(), scores.count())

> **Key idea.** `groupby` splits, applies and combines: `df.groupby("k")["c"].sum()`; named aggregation (`agg(name=("col", "func"))`) computes several summaries; group by a list for combinations; `reset_index()` turns keys back into columns. `transform` gives each row its group's value (for shares and comparisons); `df.loc[df.groupby(k)[c].idxmax()]` finds each group's top row. `merge` joins on a key, with `how` set to inner, left, right or outer; use `validate="many_to_one"` to catch duplicate keys and `indicator=True` to find unmatched rows; `concat` stacks tables. `pivot_table` goes long to wide (with `aggfunc` and `fill_value`); `melt` goes back. `pd.to_datetime` parses dates (`format`, `errors="coerce"`); `.dt` gives their parts; subtracting gives differences (`.dt.days`); `resample("MS")` totals by month. Count missing values with `isna().sum()`, then drop, fill or keep them on purpose: statistics skip `NaN`, and filling with 0 changes averages.

### Exercise: Per-group summaries

Using orders with `total` (from `load_orders`, given): write `city_revenue(df)`: a dict from city to revenue, rounded to 2 places. Then `category_table(df)`: a list of row dicts with keys `category`, `orders`, `revenue` and `average` (the mean total), rounded to 2 places, sorted by revenue from highest. And `orders_per(df, keys)`: a dict from each combination of the `keys` columns (a tuple when there are several keys, the value when there's one) to its number of orders.

In [ ]:
import pandas as pd

def load_orders(path):
    df = pd.read_csv(path)
    df["total"] = (df["quantity"] * df["unit_price"]).round(2)
    return df

def city_revenue(df):
    ___

def category_table(df):
    ___

def orders_per(df, keys):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_group():\n    \"groupby and agg\"\n    df = load_orders(\"orders.csv\")\n    assert city_revenue(df) == {\"Bath\": 71.5, \"Hull\": 98.0, \"Leeds\": 97.49, \"York\": 126.89}\n    assert category_table(df) == [\n        {\"category\": \"electronics\", \"orders\": 5, \"revenue\": 209.5, \"average\": 41.9},\n        {\"category\": \"home\", \"orders\": 5, \"revenue\": 122.48, \"average\": 24.5},\n        {\"category\": \"stationery\", \"orders\": 4, \"revenue\": 61.9, \"average\": 15.48}]\n    per = orders_per(df, [\"city\", \"category\"])\n    assert per[(\"York\", \"home\")] == 2 and per[(\"Bath\", \"electronics\")] == 2 and len(per) == 10\n    assert orders_per(df, [\"city\"])[\"York\"] == 5")

### Exercise: Each row against its group

Write `with_shares(df)`: a new DataFrame with a `city_share` column, each order's total as a fraction of its city's revenue, rounded to 3 places, and an `above_city_average` column, `True` when the order's total is above the mean total of its city. Then `top_per_city(df)`: a dict from city to the `order_id` of its biggest order.

In [ ]:
import pandas as pd

def with_shares(df):
    ___

def top_per_city(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_transform():\n    \"transform and idxmax\"\n    df = pd.read_csv(\"orders.csv\")\n    df[\"total\"] = (df[\"quantity\"] * df[\"unit_price\"]).round(2)\n    out = with_shares(df)\n    assert \"city_share\" not in df.columns, \"return a new DataFrame\"\n    assert out[\"city_share\"].tolist()[:4] == [0.059, 0.256, 0.066, 0.327]\n    assert round(out.loc[out[\"city\"] == \"York\", \"city_share\"].sum(), 2) == 1.0\n    assert out[\"above_city_average\"].tolist()[:7] == [False, False, False, False, True, True, True]\n    assert top_per_city(df) == {\"Bath\": 1008, \"Hull\": 1010, \"Leeds\": 1007, \"York\": 1012}")

### Exercise: Joining orders to customers

Write `revenue_by_tier(orders, customers)`: join each order to its customer (keeping every order, and raising an error if a customer appears twice in `customers`), then a dict from tier to revenue, rounded to 2 places. `never_ordered(orders, customers)`: the sorted names of customers with no orders. `unknown_customers(orders, customers)`: the sorted distinct names in `orders` that aren't in `customers`. `orders` already has `total`.

In [ ]:
import pandas as pd

def revenue_by_tier(orders, customers):
    ___

def never_ordered(orders, customers):
    ___

def unknown_customers(orders, customers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_join():\n    \"merge, validate and indicator\"\n    orders = pd.read_csv(\"orders.csv\")\n    orders[\"total\"] = (orders[\"quantity\"] * orders[\"unit_price\"]).round(2)\n    customers = pd.read_csv(\"customers.csv\")\n    assert revenue_by_tier(orders, customers) == {\"bronze\": 85.0, \"gold\": 146.5, \"silver\": 162.38}\n    assert never_ordered(orders, customers) == [\"Ivy Shaw\"]\n    extra = pd.concat([orders, pd.DataFrame([{\"order_id\": 1015, \"customer\": \"Zed Ray\", \"total\": 5.0}])], ignore_index=True)\n    assert unknown_customers(extra, customers) == [\"Zed Ray\"] and unknown_customers(orders, customers) == []\n    doubled = pd.concat([customers, customers.head(1)], ignore_index=True)\n    try:\n        revenue_by_tier(orders, doubled)\n    except Exception:\n        pass\n    else:\n        raise AssertionError(\"a duplicated customer should raise an error\")")

### Exercise: Long and wide

Write `month_by_category(df)`: a pivot table with one row per month (`"2026-01"` from the `date` text), one column per category, the summed totals rounded to 2 places and 0 where there were none; return it as a dict of dicts (`to_dict("index")`). Then `to_long(wide)`: for a DataFrame with a `month` column and one column per category, a list of `(month, category, total)` tuples for the **non-zero** totals, sorted.

In [ ]:
import pandas as pd

def month_by_category(df):
    ___

def to_long(wide):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_reshape():\n    \"pivot_table and melt\"\n    df = pd.read_csv(\"orders.csv\")\n    df[\"total\"] = (df[\"quantity\"] * df[\"unit_price\"]).round(2)\n    table = month_by_category(df)\n    assert table[\"2026-01\"] == {\"electronics\": 0.0, \"home\": 56.99, \"stationery\": 15.9}\n    assert table[\"2026-03\"][\"electronics\"] == 104.0 and table[\"2026-04\"][\"home\"] == 0\n    wide = pd.DataFrame({\"month\": [\"2026-01\", \"2026-02\"], \"home\": [56.99, 0.0], \"toys\": [3.5, 12.0]})\n    assert to_long(wide) == [(\"2026-01\", \"home\", 56.99), (\"2026-01\", \"toys\", 3.5), (\"2026-02\", \"toys\", 12.0)]")

### Exercise: Working with dates

Write `monthly_revenue(df)`: total revenue per month as a list of `("2026-01", amount)` pairs (amounts rounded to 2 places), using `pd.to_datetime` and `resample("MS")`. `busiest_weekday(df)`: the day name with most orders. `days_to_first_order(orders, customers)`: a dict from customer to the days between their `joined` date and their **first** order (customers who never ordered are left out).

In [ ]:
import pandas as pd

def monthly_revenue(df):
    ___

def busiest_weekday(df):
    ___

def days_to_first_order(orders, customers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_dates():\n    \"to_datetime, dt and resample\"\n    orders = pd.read_csv(\"orders.csv\")\n    orders[\"total\"] = (orders[\"quantity\"] * orders[\"unit_price\"]).round(2)\n    customers = pd.read_csv(\"customers.csv\")\n    assert monthly_revenue(orders) == [(\"2026-01\", 72.89), (\"2026-02\", 142.0), (\"2026-03\", 163.49), (\"2026-04\", 15.5)]\n    assert busiest_weekday(orders) == \"Sunday\"\n    waits = days_to_first_order(orders, customers)\n    assert waits[\"Ada Lovelace\"] == 63 and waits[\"Cy Okafor\"] == 1 and waits[\"Dee Patel\"] == 203\n    assert \"Ivy Shaw\" not in waits and len(waits) == 8\n    assert orders[\"date\"].iloc[0] == \"2026-01-04\", \"leave the original columns as they were\"")

### Exercise: Missing on purpose

A survey table has columns `name`, `age`, `score` and `comment`, with gaps. Write `clean_survey(df)`: drop rows with no `score` (they can't be used), fill missing `comment`s with `""`, and leave missing ages alone; return the new DataFrame with the index renumbered. Then `average_age(df)`: the mean age of the people whose age is known, rounded to 1 place, and `missing_counts(df)`: a dict of the columns that have gaps, to how many.

In [ ]:
import pandas as pd

def clean_survey(df):
    ___

def average_age(df):
    ___

def missing_counts(df):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_missing():\n    \"drop, fill or keep\"\n    df = pd.DataFrame({\"name\": [\"Ada\", \"Bo\", \"Cy\", \"Dee\", \"Eli\"], \"age\": [34, None, 51, None, 23],\n                       \"score\": [8, 6, None, 9, 7], \"comment\": [\"great\", None, \"ok\", None, \"fine\"]})\n    assert missing_counts(df) == {\"age\": 2, \"score\": 1, \"comment\": 2}\n    out = clean_survey(df)\n    assert out[\"name\"].tolist() == [\"Ada\", \"Bo\", \"Dee\", \"Eli\"] and out.index.tolist() == [0, 1, 2, 3]\n    assert out[\"comment\"].tolist() == [\"great\", \"\", \"\", \"fine\"] and int(out[\"age\"].isna().sum()) == 2\n    assert average_age(df) == 36.0 and average_age(out) == 28.5\n    assert len(df) == 5")

### Bug hunt: Revenue that grew by itself

The customers file was exported twice for one person, so the join duplicated her orders and the revenue for gold customers is inflated: it prints `{'gold': 30.0, 'silver': 5.0}`. Remove the duplicate customer rows before joining so it prints `{'gold': 20.0, 'silver': 5.0}`.

In [ ]:
import pandas as pd

orders = pd.DataFrame({"customer": ["Ada", "Ada", "Bo"], "total": [12.0, 8.0, 5.0]})
customers = pd.DataFrame({"customer": ["Ada", "Bo", "Ada"], "tier": ["gold", "silver", "gold"]})
joined = orders.merge(customers, on="customer")
print(joined.groupby("tier")["total"].sum().to_dict())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one row per customer\"\n    expect_output(\"{'gold': 20.0, 'silver': 5.0}\")")

### Bug hunt: March before January

The dates are day/month/year text, so sorting them sorts the text: this prints `['02/03/2026', '15/01/2026', '28/02/2026']`. Parse them as dates (with the right format) and sort by those, so it prints `['15/01/2026', '28/02/2026', '02/03/2026']`.

In [ ]:
import pandas as pd

df = pd.DataFrame({"when": ["15/01/2026", "02/03/2026", "28/02/2026"]})
print(df.sort_values("when")["when"].tolist())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"real dates\"\n    expect_output(\"['15/01/2026', '28/02/2026', '02/03/2026']\")")

### Bug hunt: An average dragged down

Two people didn't answer, and filling their gaps with 0 makes the average rating look poor: `average: 4.0`. The missing answers shouldn't count, so it should print `average: 8.0`.

In [ ]:
import pandas as pd

ratings = pd.Series([9, None, 7, None, 8, 8, 9, 7])
print("average:", round(float(ratings.fillna(0).mean()), 1))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the gaps out\"\n    expect_output(\"average: 8.0\")")

### Puzzle: Revenue per tier, biggest first

Arrange the lines to join orders to customers safely, total the revenue per tier and print the tiers from biggest to smallest. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import pandas as pd

df = pd.DataFrame({"k": ["a", "b", "a", "b", "a"], "v": [1, 2, 3, 4, 5]})
p = ___   # df.groupby("k")["v"].sum().to_dict()
q = ___   # df.groupby("k")["v"].transform("max").tolist()
r = ___   # len(df.merge(pd.DataFrame({"k": ["a"]}), on="k"))
s = ___   # len(df.merge(pd.DataFrame({"k": ["a", "c"]}), on="k", how="outer"))
t = ___   # pd.Series([2.0, None, 4.0]).mean()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == {\"a\": 9, \"b\": 6} and q == [5, 4, 5, 4, 5], \"p and q\"\n    assert r == 3 and s == 6 and t == 3.0, \"r, s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the monthly sales report

Write `sales_report(orders, customers)` for the shop's two tables (`orders` without a `total` column; compute it as `quantity × unit_price` rounded to 2 places). Return a dict:

- `"monthly"`: a list of row dicts, one per month in order, with `month` (`"2026-01"`), `revenue` (2 places), `orders`, `customers` (distinct) and `growth` (the fractional change in revenue from the previous month, 3 places, `None` for the first);
- `"top_customer"`: a dict from month to the customer who spent most that month (ties: alphabetical first);
- `"tier_category"`: a pivot of revenue with tiers as rows and categories as columns, 0 where empty, as a dict of dicts, 2 places;
- `"repeat_rate"`: the fraction of customers **who ordered** who ordered more than once, 3 places;
- `"inactive"`: sorted names of customers with no orders.

Join safely (every order kept, duplicate customers an error). Don't change either input.

In [ ]:
import pandas as pd

def sales_report(orders, customers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef _report():\n    orders = pd.read_csv(\"orders.csv\")\n    customers = pd.read_csv(\"customers.csv\")\n    cols = orders.columns.tolist()\n    report = sales_report(orders, customers)\n    assert orders.columns.tolist() == cols, \"don't change the inputs\"\n    return report\n\ndef test_monthly():\n    \"one row per month\"\n    m = _report()[\"monthly\"]\n    assert [r[\"month\"] for r in m] == [\"2026-01\", \"2026-02\", \"2026-03\", \"2026-04\"]\n    assert m[0] == {\"month\": \"2026-01\", \"revenue\": 72.89, \"orders\": 4, \"customers\": 4, \"growth\": None}\n    assert m[1][\"growth\"] == 0.948 and m[3][\"growth\"] == -0.905 and m[2][\"revenue\"] == 163.49 and m[2][\"customers\"] == 5\n\ndef test_the_rest():\n    \"customers, tiers and repeats\"\n    r = _report()\n    assert r[\"top_customer\"] == {\"2026-01\": \"Dee Patel\", \"2026-02\": \"Bo Chen\", \"2026-03\": \"Ada Lovelace\", \"2026-04\": \"Eli Novak\"}\n    assert r[\"tier_category\"][\"gold\"] == {\"electronics\": 59.0, \"home\": 59.0, \"stationery\": 28.5}\n    assert r[\"tier_category\"][\"bronze\"][\"home\"] == 13.5\n    assert r[\"repeat_rate\"] == 0.625 and r[\"inactive\"] == [\"Ivy Shaw\"]")

## Warm-up

`groupby("city")` splits the rows into one group per city; `["total"].sum()` adds up each group's totals; the result is a Series whose **index is the cities**. That's the pattern called **split, apply, combine**, and it answers most "per something" questions in one line.

## Grouping

- `df.groupby("key")["col"].sum()` (or `mean`, `count`, `size`, `max`, `nunique`, ...) gives one value per group.
- Several summaries at once with **named aggregation**: `df.groupby("key").agg(orders=("order_id", "count"), revenue=("total", "sum"))`, each new column naming `(source column, function)`.
- Group by several keys with a list: `groupby(["city", "category"])`. The result has one row per **combination** that occurs.
- The keys become the index; `.reset_index()` (or `as_index=False`) turns them back into ordinary columns, which is easier to sort and print.
- **`transform`** returns a result **for every row** (each row gets its group's value), so you can compare a row with its group: `df["total"] / df.groupby("city")["total"].transform("sum")` is each order's share of its city.
- `df.loc[df.groupby("city")["total"].idxmax()]` is each group's top row.

## Joining tables

Data usually comes in several tables that share a **key**: orders name a customer, and a customers table describes each one. **`orders.merge(customers, on="customer")`** joins them: every order gets its customer's columns. Which rows survive depends on **`how`**:

- `"inner"` (the default): only keys found in **both** tables;
- `"left"`: every row of the left table; missing matches get `NaN`;
- `"right"` and `"outer"`: every row of the right table, or of both.

Two habits prevent the classic join bugs. **`validate="many_to_one"`** makes pandas raise an error if the right table has a key more than once (which would quietly duplicate rows and inflate every total). And **`indicator=True`** adds a `_merge` column saying `"both"`, `"left_only"` or `"right_only"`, which is how you find the customers who never ordered, or orders for customers you don't know. When the key columns have different names, use `left_on=` and `right_on=`. To stack tables with the same columns (January's orders, then February's), use `pd.concat([a, b], ignore_index=True)`.

## Reshaping

Data is **long** when each row is one observation (an order, with its month and category) and **wide** when one row holds many (a month, with a column per category). Reports and charts usually want wide; storage and grouping usually want long.

- **`df.pivot_table(index="month", columns="category", values="total", aggfunc="sum", fill_value=0)`** goes from long to wide, summarising as it goes (pivot tables in spreadsheets work the same way).
- **`wide.melt(id_vars="month", var_name="category", value_name="total")`** goes back from wide to long.

## Dates

Dates in a CSV arrive as text. **`pd.to_datetime(df["date"])`** turns them into real dates (give `format="%d/%m/%Y"` when they aren't in the `2026-03-15` style, and `errors="coerce"` to turn unparseable ones into `NaT`, "not a time"). Then the **`.dt`** accessor gives their parts: `.dt.year`, `.dt.month`, `.dt.day_name()`, `.dt.strftime("%Y-%m")`. Subtracting dates gives time differences, and `.dt.days` turns those into whole days.

For totals per day, week or month, make the dates the index and **`resample`**: `df.set_index("date").resample("MS")["total"].sum()` gives one row per month (`"MS"` means "month start"; `"W"` is weekly, `"D"` daily), **including** months with no orders.

`pct_change` is each month's change from the previous one as a fraction; the first has nothing to compare with, so it's `NaN`.

## Missing values

pandas marks missing values as `NaN` (or `NaT` for dates, and `None` may appear in text). Before fixing them, count them: `df.isna().sum()` per column. Then decide, column by column, what missing **means**:

- **drop** rows that can't be used: `df.dropna(subset=["email"])`;
- **fill** with a sensible value: `df.fillna({"discount": 0})`, or a column's median;
- **keep** them: most statistics skip `NaN` already (`mean`, `sum`, `count`), which is usually right; filling with 0 first would drag an average down.

Each choice changes the answers, so write down which you made. And chain the steps into one readable pipeline with methods that return new DataFrames: `df.dropna(...).assign(...).query(...)`.

## Practice

## Quiz

## Daily challenge

## Before you go

Take the CSV you explored yesterday and ask it three "per something" questions with `groupby`, one with a `pivot_table`, and, if it has dates, one with `resample`. If you can find a second table that shares a column with it, join them and check the row count before and after: if it grew, find out why. Tomorrow is **Visualisation with matplotlib**: turning these tables into charts that tell the truth.